# Samsung GenAI Lab
# Mission: Build a Transformer Next-Word Predictor

## Learning Outcomes
- Understand Tokenization
- Understand Next Token Prediction
- Compare Transformer Models
- Build a simple Gradio UI


## Mission 1
Run the next cell to install libraries.
**Checkpoint:** No errors should appear.


In [ ]:
!pip -q install transformers torch sentencepiece accelerate gradio

## Mission 2 - Imports
**TODO:** Run the cell.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import gradio as gr


## Mission 3 - Choose ONE Model

Uncomment ONLY ONE model.
Compare output quality with your classmates.


In [ ]:
MODEL_NAME="HuggingFaceTB/SmolLM2-360M-Instruct"

# MODEL_NAME="distilgpt2"

# MODEL_NAME="Qwen/Qwen2.5-0.5B-Instruct"


## Mission 4 - Load Model

Run this cell. It downloads the model weights — may take **1–2 minutes** the first time.

You will see a progress bar. Wait for **Model Loaded** to appear before moving to Mission 5.

In [ ]:
# Load the tokenizer — converts words to token IDs and back
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load the model weights from HuggingFace Hub
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

print("Model Loaded")

## Mission 5 - Tokenization Investigation

Questions:
1. How many tokens are created?
2. Print the token IDs.
3. What is the vocabulary size?


In [ ]:
prompt="Artificial Intelligence will"

inputs=tokenizer(prompt,return_tensors="pt")

print("Input IDs:",inputs["input_ids"])
print("Number of Tokens:",len(inputs["input_ids"][0]))
print("Vocabulary Size:",tokenizer.vocab_size)


## Mission 6 - Next Token Prediction

This is the core of how a transformer predicts text. Three steps happen inside:

| Step | What it does |
|---|---|
| **Logits** | The model outputs one raw score per word in the vocabulary (~32,000 scores). Higher = more likely next word. |
| **Softmax** | Converts raw scores into probabilities that add up to 1.0 |
| **Top-K** | Picks the K highest probability words — these are the predictions |

Read each comment in the code below, then run the cell.

In [ ]:
def predict_next_words(prompt, top_k=5):

    # Convert the prompt text into token IDs the model understands
    inputs = tokenizer(prompt, return_tensors='pt')

    # Run the model — torch.no_grad() saves memory (no training needed)
    with torch.no_grad():
        outputs = model(**inputs)

    # Get the logits for the LAST token position only
    # [:, -1, :] means: all batches, last token, all vocab scores
    logits = outputs.logits[:, -1, :]

    # Convert logits to probabilities — all values now sum to 1.0
    probs = torch.softmax(logits, dim=-1)

    # Pick the top_k highest probability tokens
    # values = their probabilities, indices = their IDs in vocabulary
    values, indices = torch.topk(probs, top_k)

    results = []
    for score, idx in zip(values[0], indices[0]):
        token = tokenizer.decode([idx]).strip()  # Convert ID back to word
        results.append((token, float(score)))

    return results

## Mission 7 — Your Own Prompts

**TODO — you write this part.**

Find the three variables below and **replace all three** with your own sentence starters.

**Rules:**
- Each prompt must be at least 4 words
- Try one technical topic, one everyday topic, one of your own choice
- Do not leave the default examples unchanged

**Checkpoint:** After running, you should see **15 predictions** — 5 per prompt. Discuss with your team: which prediction surprised you most?

In [ ]:
# TODO: Replace ALL THREE prompts below with your own sentence starters
# Rules: at least 4 words each -- do NOT leave the default examples

my_prompt_1 = "Artificial Intelligence will"   # <- CHANGE THIS
my_prompt_2 = "The future of technology"       # <- CHANGE THIS
my_prompt_3 = "Samsung engineers are"          # <- CHANGE THIS

# Runs all 3 prompts and prints top-5 predictions for each
print("=" * 45)
for prompt in [my_prompt_1, my_prompt_2, my_prompt_3]:
    print(f"\nPrompt: '{prompt}'")
    results = predict_next_words(prompt)
    for i, (w, s) in enumerate(results, 1):
        print(f"  {i}. {w}  ({s:.4f})")
print("=" * 45)
print("\nDiscuss: which prediction surprised your team most?")

## Mission 8 - Gradio UI

Build an interactive web app for your predictor. Run the cell — it generates a public URL valid for 72 hours.

**Copy the `gradio.live` URL** — you need it for your day1.json submission.

In [ ]:
def app(prompt):
    out=""
    for i,(w,s) in enumerate(predict_next_words(prompt),1):
        out+=f"{i}. {w} ({s:.4f})\n"
    return out

demo=gr.Interface(
fn=app,
inputs=gr.Textbox(label="Prompt"),
outputs=gr.Textbox(label="Top 5 Next Word Predictions"),
title="Samsung GenAI - Transformer Lab",
examples=[
["Artificial Intelligence will"],
["Samsung is developing"],
["Cloud computing enables"],
["Transformers work by"]
]
)

demo.launch(share=True)


## Done!

Your team has:
- Built a transformer next-word predictor
- Compared model outputs across teams
- Launched a live Gradio demo

**Before Lab 3:** Note down your Gradio URL and your team name.
You will add these to day1.json when submitting after Lab 3.

Be ready to present for 2 minutes during the **Gradio Showdown**:
- Which model you chose and why
- Your most interesting prompt result
- What you learned about logits, softmax, and top-k